# Milestone2 — definitive Colab GPU worker
Floating memories • 1080p24 • 12 samples • ProRes 422 + 16-bit depth. Select a GPU runtime and Run all. Authorize the Google account with write access to the linked job folder. This notebook uses Drive folder IDs rather than ambiguous mounted paths. Run one worker per job.

In [1]:
from google.colab import auth
auth.authenticate_user()
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
import os, json, hashlib, shutil, io, time, subprocess, tarfile
creds,_=default()
drive_api=build('drive','v3',credentials=creds,cache_discovery=False)
JOB_ID='1qYdX-8ia0THTt0KFI5qzKBjt7xNJjA13'
JOB_JSON_ID='1fztia838tPSsw3c-_j4zpkmc2-sBr2Wq'
INPUT_ID='14Bq_F6WTdkIAb6DLd7ljcR72fj-wdFwV'
folder=drive_api.files().get(fileId=JOB_ID,fields='id,name,capabilities(canAddChildren)',supportsAllDrives=True).execute()
assert folder['capabilities']['canAddChildren'], 'Authorize an account with write access to the job folder'
print('Verified Drive job:',folder['name'],folder['id'])
def children(parent):
    q=f"'{parent}' in parents and trashed=false"
    token=None; result=[]
    while True:
        r=drive_api.files().list(q=q,fields='nextPageToken,files(id,name,size,md5Checksum,mimeType)',pageSize=1000,pageToken=token,supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
        result.extend(r.get('files',[]))
        token=r.get('nextPageToken')
        if not token:return result
def lookup(parent,name):
    matches=[x for x in children(parent) if x['name']==name]
    assert len(matches)<=1, f'Duplicate Drive name: {name}'
    return matches[0] if matches else None
def folder_id(parent,name):
    f=lookup(parent,name)
    if f:
        assert f['mimeType']=='application/vnd.google-apps.folder'
        return f['id']
    return drive_api.files().create(body={'name':name,'parents':[parent],'mimeType':'application/vnd.google-apps.folder'},fields='id',supportsAllDrives=True).execute()['id']
def download(fid,dst):
    dst.parent.mkdir(parents=True,exist_ok=True)
    tmp=dst.with_suffix(dst.suffix+'.downloading')
    with tmp.open('wb') as f:
        loader=MediaIoBaseDownload(f,drive_api.files().get_media(fileId=fid,supportsAllDrives=True),chunksize=32*1024*1024)
        done=False
        while not done:
            status,done=loader.next_chunk(num_retries=5)
    os.replace(tmp,dst)
def upload(src,parent,existing=None):
    with src.open('rb') as f: md5=hashlib.file_digest(f,'md5').hexdigest()
    if existing and existing.get('md5Checksum')==md5 and int(existing.get('size',-1))==src.stat().st_size:return existing
    media=MediaFileUpload(str(src),resumable=True,chunksize=8*1024*1024)
    args=dict(media_body=media,fields='id,name,size,md5Checksum',supportsAllDrives=True)
    if existing:req=drive_api.files().update(fileId=existing['id'],**args)
    else:req=drive_api.files().create(body={'name':src.name,'parents':[parent]},**args)
    result=None
    while result is None: _,result=req.next_chunk(num_retries=5)
    assert result['md5Checksum']==md5 and int(result['size'])==src.stat().st_size, 'Upload verification failed'
    return result
cfg=json.loads(drive_api.files().get_media(fileId=JOB_JSON_ID,supportsAllDrives=True).execute())
assert cfg['version']==2 and cfg['scene']=='floating' and cfg['count']==2420
print('Job:',cfg['scene'],cfg['count'],'frames')


Verified Drive job: floating-prores-v1 1qYdX-8ia0THTt0KFI5qzKBjt7xNJjA13
Job: floating 2420 frames


In [2]:
import pathlib, subprocess, urllib.request, hashlib, tarfile, json, shutil
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
base='https://download.blender.org/release/Blender5.0/'
name='blender-5.0.1-linux-x64.tar.xz'
archive=Path('/content')/name
exe=Path('/content/blender-5.0.1-linux-x64/blender')
if not exe.is_file():
    urllib.request.urlretrieve(base+name,archive)
    expected=next(line.split()[0] for line in urllib.request.urlopen(base+'blender-5.0.1.sha256').read().decode().splitlines() if line.endswith(name))
    with archive.open('rb') as f: assert hashlib.file_digest(f,'sha256').hexdigest()==expected, 'Blender checksum mismatch'
    subprocess.run(['tar','-xJf',str(archive),'-C','/content'],check=True)

import shutil
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'FFmpeg required'


In [3]:
work=Path('/content/milestone2-worker')/cfg['sha256'][:16]
work.mkdir(parents=True,exist_ok=True)
bundle=work/'input.tar'
def digest(path):
    with path.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
if not bundle.exists() or digest(bundle)!=cfg['sha256']:
    print('Downloading verified geometry package (2.32 GB)',flush=True)
    download(INPUT_ID,bundle)
assert digest(bundle)==cfg['sha256'], 'Input checksum mismatch'
with tarfile.open(bundle) as t:t.extractall(work,filter='data')
meta=json.loads((work/'cache/manifest.json').read_text())
assert meta['count']==cfg['count'] and meta['settings']['buildHash']==cfg['buildHash']
out=work/'render.prores';out.mkdir(exist_ok=True)
remote=folder_id(JOB_ID,'render.prores')
chunks_id=folder_id(remote,'chunks')
depth_id=folder_id(remote,'depth')
signature=out/'job-settings.json'
signature.write_text(json.dumps(cfg,sort_keys=True))
old=lookup(remote,signature.name)
if old:
    assert json.loads(drive_api.files().get_media(fileId=old['id'],supportsAllDrives=True).execute())==cfg,'Changed job settings'
else:upload(signature,remote)
remote_chunks={x['name']:x for x in children(chunks_id)}
remote_depth={x['name']:x for x in children(depth_id)}
for name,mark in sorted(remote_chunks.items()):
    if not name.endswith('.json'):continue
    dst=out/'chunks'/name
    download(mark['id'],dst)
    d=json.loads(dst.read_text())
    mov=dst.with_suffix('.mov')
    download(remote_chunks[mov.name]['id'],mov)
    for i in range(d['start'],d['start']+d['count']):
        p=out/'depth'/f'frame-{i:06d}.png'
        download(remote_depth[p.name]['id'],p)
print('Verified cache and restored committed chunks',flush=True)


Verified cache and restored committed chunks


In [13]:
from pathlib import Path

renderer = work / "blender/render_cache.py"
source = renderer.read_text()

old = 'scene.cycles.denoiser="OPTIX" if a.device=="OPTIX" else "OPENIMAGEDENOISE"'

new = '''scene.cycles.denoiser="OPENIMAGEDENOISE"
    scene.cycles.denoising_use_gpu=True'''

if old in source:
    renderer.write_text(source.replace(old, new))
    print("Patched successfully")
elif new in source:
    print("Already patched")
else:
    raise RuntimeError("Renderer configuration not found")

Already patched


In [ ]:
import subprocess
import threading
import time
import shutil

DEVICE = "OPTIX"

# Verify that OptiX is available before rendering.
subprocess.run([
    str(exe),
    "-b",
    "--factory-startup",
    "--python-exit-code", "1",
    "--python-expr",
    """
import bpy

p = bpy.context.preferences.addons['cycles'].preferences
p.compute_device_type = 'OPTIX'
p.refresh_devices()

devices = [d for d in p.devices if d.type == 'OPTIX']
assert devices, 'No OptiX GPU detected'

for d in devices:
    print('OptiX GPU:', d.name)
"""
], check=True)


# Upload completed chunks to Google Drive.
def sync_chunks():
    for mark in sorted((out / "chunks").glob("chunk-*.json")):
        if mark.name in remote_chunks:
            continue

        d = json.loads(mark.read_text())

        for i in range(d["start"], d["start"] + d["count"]):
            p = out / "depth" / f"frame-{i:06d}.png"

            remote_depth[p.name] = upload(
                p,
                depth_id,
                remote_depth.get(p.name)
            )

        mov = mark.with_suffix(".mov")

        remote_chunks[mov.name] = upload(
            mov,
            chunks_id,
            remote_chunks.get(mov.name)
        )

        remote_chunks[mark.name] = upload(mark, chunks_id)

        print(
            f"[DRIVE] Uploaded chunk {d['start']} "
            f"({d['count']} frames)",
            flush=True
        )


# Build rendering command.
cmd = [
    str(exe),
    "-b",
    "--factory-startup",
    "--python-exit-code", "1",
    "-P", str(work / "blender/render_cache.py"),
    "--",
    "--cache", str(work / "cache"),
    "--output", str(out),
    "--format", "PNG",
    "--passes", "beauty",
    "--prores",
    "--device", DEVICE,
    "--ffmpeg", shutil.which("ffmpeg"),
    "--ffprobe", shutil.which("ffprobe"),
]

for k in (
    "width",
    "height",
    "samples",
    "bounces",
    "noise",
    "min_samples",
    "diffuse",
    "glossy",
    "chunk_size",
    "depth_far"
):
    cmd += [
        "--" + k.replace("_", "-"),
        str(cfg[k])
    ]


print("=" * 60)
print("MILESTONE2 — OPTIX RENDER")
print("=" * 60)
print("Device:", DEVICE)
print("Samples:", cfg["samples"])
print("Bounces:", cfg["bounces"])
print("Resolution:", cfg["width"], "x", cfg["height"])
print("Output: ProRes 422 + 16-bit depth PNG")
print("=" * 60)


# Start Blender with live logging.
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


def stream_logs():
    for line in proc.stdout:
        print("BLENDER:", line.rstrip(), flush=True)


log_thread = threading.Thread(
    target=stream_logs,
    daemon=True
)

log_thread.start()

try:
    while proc.poll() is None:
        sync_chunks()
        time.sleep(5)

finally:
    if proc.poll() is None:
        proc.terminate()
        proc.wait()

    log_thread.join(timeout=10)
    sync_chunks()


assert proc.returncode == 0, (
    "Render interrupted. Rerun this cell to resume."
)

for name in ("depth-metadata.json", "media.json"):
    upload(
        out / name,
        remote,
        lookup(remote, name)
    )

print("=" * 60)
print("RENDER COMPLETE")
print("https://drive.google.com/drive/folders/" + remote)
print("=" * 60)

MILESTONE2 — OPTIX RENDER
Device: OPTIX
Samples: 12
Bounces: 3
Resolution: 1920 x 1080
Output: ProRes 422 + 16-bit depth PNG
BLENDER: 00:00.847  cycles           | WARNING HIPEW initialization failed: Error opening HIP dynamic library
BLENDER: [cycles] GPU: Tesla T4
BLENDER: /content/milestone2-worker/10233c239b2295f8/blender/render_cache.py:94: DeprecationWarning: 'World.use_nodes' is expected to be removed in Blender 6.0
BLENDER:   scene.world.use_nodes=True
BLENDER: /content/milestone2-worker/10233c239b2295f8/blender/render_cache.py:122: DeprecationWarning: 'Material.use_nodes' is expected to be removed in Blender 6.0
BLENDER:   mat=bpy.data.materials.new("Material-%d"%m["id"]);mat.use_nodes=True
BLENDER: [cycles] Mesh datablocks: 289 for 720 objects
BLENDER: [cycles] 720 meshes; 27 materials; 2420 frames
BLENDER: [cycles] Replay starting at frame 0
BLENDER: [cycles] Resume 1
BLENDER: [cycles] Resume 2
BLENDER: [cycles] Resume 3
BLENDER: [cycles] Resume 4
BLENDER: [cycles] Resume 5
